# 01 — Simple GRU Baseline

Train a lightweight single-layer GRU toxicity classifier on the Jigsaw dataset, then use `ModelBiasEvaluator` to measure model-level bias on the held-out test set.

| § | Step |
|---|------|
| 1 | Load & split data (annotated subset) |
| 2 | Train a BPE subword tokenizer |
| 3 | Define a GRU classifier |
| 4 | Train for 3 epochs |
| 5 | Overall performance |
| 6 | Fairness evaluation — Subgroup AUC, FPR, ECE |
| 7 | Counterfactual gap analysis |

> **Authorship note.** Code generation and prose editing in this notebook were assisted by Claude Opus. Method selection, limitation analysis, and the substance of result interpretations were completed by the human author.

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, roc_auc_score
from transformers import BertTokenizer

from fairness_jigsaw.metrics import DEFAULT_IDENTITY_COLUMNS, ModelBiasEvaluator

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── data ──────────────────────────────────────────────────────────────────────────────
TOXICITY_THRESHOLD = 0.5
IDENTITY_THRESHOLD = 0.5
N_TRAIN_SAMPLE     = None  # set to int (e.g. 200_000) to subsample train for speed

# ── tokenizer ────────────────────────────────────────────────────────────────────────
MAX_LEN    = 128

# ── model ─────────────────────────────────────────────────────────────────────────────
EMBED_DIM  = 64
HIDDEN_DIM = 128

# ── training ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 256
EPOCHS     = 3
LR         = 1e-3

Device: mps


## 1. Data

Use the pre-defined splits in `data/split_ids.json` (seed 1337, 80/10/10 split of the full 1.8M-row corpus). The test set is used for both overall metrics and fairness evaluation — `ModelBiasEvaluator` automatically restricts each subgroup metric to rows that carry identity annotations.

Set `N_TRAIN_SAMPLE` in §0 to an integer to subsample the training set for faster iteration.

In [2]:
with open("../data/split_ids.json") as f:
    split_ids = json.load(f)

USE_COLS = ["id", "target", "comment_text"] + list(DEFAULT_IDENTITY_COLUMNS)
df_all = pd.read_csv("../data/train.csv", usecols=USE_COLS)
df_all["toxic"] = (df_all["target"] >= TOXICITY_THRESHOLD).astype(int)

train_df = df_all[df_all["id"].isin(split_ids["train"])].reset_index(drop=True)
val_df   = df_all[df_all["id"].isin(split_ids["val"])].reset_index(drop=True)
test_df  = df_all[df_all["id"].isin(split_ids["test"])].reset_index(drop=True)

if N_TRAIN_SAMPLE is not None:
    train_df = train_df.sample(n=N_TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)

for name, df in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    anno = df[list(DEFAULT_IDENTITY_COLUMNS)].notna().any(axis=1).sum()
    print(f"{name}: {len(df):>10,}  | toxic: {df['toxic'].mean():.2%}"
          f"  | annotated: {anno:>7,} ({anno/len(df):.1%})")

Train:  1,443,897  | toxic: 8.00%  | annotated: 324,097 (22.4%)
Val  :    180,486  | toxic: 8.00%  | annotated:  40,486 (22.4%)
Test :    180,491  | toxic: 8.00%  | annotated:  40,547 (22.5%)


## 2. Text Preprocessing

Load a **BERT WordPiece** tokenizer (`bert-base-uncased`). Sequences are truncated or padded to `MAX_LEN`.

In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print(f"BERT vocabulary size : {tokenizer.vocab_size:,} tokens")
print(f"Example tokens       : {tokenizer.tokenize('I hate this person')}")


def encode(text) -> list[int]:
    return tokenizer.encode(
        text if isinstance(text, str) else "",
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )


class CommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y = torch.tensor(df["toxic"].values, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_loader = DataLoader(CommentDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(CommentDataset(test_df),  batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

BERT vocabulary size : 30,522 tokens
Example tokens       : ['i', 'hate', 'this', 'person']
Train batches: 5641  |  Test batches: 706


## 3. Model

Single-layer GRU classifier. The final hidden state is passed through a linear head followed by sigmoid to produce a toxicity probability in [0, 1].

```
Embedding(|V|, d_e)  →  GRU(d_e, d_h)  →  Linear(d_h, 1)  →  Sigmoid
```

In [4]:
class ToxicityGRU(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru  = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)            # (B, L, E)
        _, h = self.gru(x)               # h: (1, B, H)
        return torch.sigmoid(self.head(h[-1])).squeeze(-1)  # (B,)


model    = ToxicityGRU(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {n_params:,}")

ToxicityGRU(
  (embedding): Embedding(30522, 64, padding_idx=0)
  (gru): GRU(64, 128, batch_first=True)
  (head): Linear(in_features=128, out_features=1, bias=True)
)

Total parameters: 2,028,033


## 4. Training

Binary cross-entropy with Adam.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    print(f"Epoch {epoch}/{EPOCHS}  avg_loss={total_loss / n_batches:.4f}")

Epoch 1/3  avg_loss=0.1644
Epoch 2/3  avg_loss=0.1281
Epoch 3/3  avg_loss=0.1203


## 5. Inference & Overall Performance

In [6]:
model.eval()
raw_scores: list[float] = []
with torch.no_grad():
    for x, _ in test_loader:
        raw_scores.extend(model(x.to(DEVICE)).cpu().tolist())

test_df = test_df.copy()
test_df["score"] = raw_scores

overall_auc = roc_auc_score(test_df["toxic"], test_df["score"])
pred_labels  = (test_df["score"] >= TOXICITY_THRESHOLD).astype(int)

print(f"Overall AUC          : {overall_auc:.4f}")
print(f"Predicted toxic rate : {pred_labels.mean():.2%}")
print(f"True toxic rate      : {test_df['toxic'].mean():.2%}\n")
print(classification_report(test_df["toxic"], pred_labels, target_names=["non-toxic", "toxic"]))

Overall AUC          : 0.9540
Predicted toxic rate : 5.30%
True toxic rate      : 8.00%

              precision    recall  f1-score   support

   non-toxic       0.96      0.99      0.97    166056
       toxic       0.80      0.53      0.63     14435

    accuracy                           0.95    180491
   macro avg       0.88      0.76      0.80    180491
weighted avg       0.95      0.95      0.95    180491



## 6. Fairness Evaluation

`ModelBiasEvaluator` computes three complementary subgroup metrics on the test set.

| Metric | What it captures |
|--------|-----------------|
| **Subgroup / BPSN / BNSP / Pinned AUC** | Discrimination quality for and against each identity |
| **Subgroup FPR (+ gap vs background)** | Over-flagging of non-toxic comments per identity |
| **ECE** | Calibration quality — how aligned are model confidence scores with true probabilities |

Tables are sorted worst-first.

In [7]:
evaluator = ModelBiasEvaluator(
    identity_cols=DEFAULT_IDENTITY_COLUMNS,
    toxicity_threshold=TOXICITY_THRESHOLD,
    identity_threshold=IDENTITY_THRESHOLD,
    min_subgroup_size=20,
)

results = evaluator.evaluate(test_df, score_col="score", label_col="toxic")

### AUC Metrics

Sorted by pinned AUC ascending (worst subgroups first). Low BPSN AUC → model over-predicts toxicity for the group; low BNSP AUC → under-predicts.

In [8]:
results["auc"].round(4)

,identity,n,subgroup_auc,bpsn_auc,bnsp_auc,pinned_auc
0,other_religion,34,0.7172,0.9104,0.8798,0.8070
1,hindu,55,0.7287,0.8926,0.8929,0.8140
2,black,1496,0.8110,0.8062,0.9640,0.8444
3,homosexual_gay_or_lesbian,1141,0.8266,0.8056,0.9697,0.8514
4,white,2551,0.8307,0.8227,0.9655,0.8601
5,muslim,2151,0.8278,0.8401,0.9592,0.8650
6,buddhist,49,0.8101,0.8786,0.9387,0.8663
7,transgender,255,0.8317,0.8448,0.9587,0.8685
8,other_race_or_ethnicity,42,0.8367,0.8689,0.9430,0.8765
9,heterosexual,116,0.8532,0.8770,0.9474,0.8875


### Subgroup FPR

FPR = fraction of truly non-toxic comments in the subgroup that the model flags as toxic. Positive `fpr_gap` = the model over-triggers on that identity relative to background.

In [9]:
results["fpr"].round(4)

,identity,n_negatives,fpr,bg_fpr,fpr_gap
0,black,1039,0.0703,0.0114,0.0588
1,transgender,206,0.0631,0.0117,0.0514
2,homosexual_gay_or_lesbian,819,0.0623,0.0116,0.0507
3,other_race_or_ethnicity,35,0.0571,0.0118,0.0453
4,white,1839,0.0457,0.0114,0.0343
5,hindu,47,0.0426,0.0118,0.0308
6,psychiatric_or_mental_illness,365,0.0411,0.0117,0.0294
7,atheist,108,0.0370,0.0118,0.0253
8,muslim,1662,0.0337,0.0116,0.0221
9,heterosexual,91,0.0330,0.0118,0.0212


### Expected Calibration Error

First row is the overall ECE. Subgroup ECE > overall ECE signals the model is less well-calibrated for that group.

In [10]:
results["ece"].round(4)

,identity,n,ece
0,heterosexual,116,0.1146
1,other_religion,34,0.1098
2,hindu,55,0.0879
3,other_race_or_ethnicity,42,0.0806
4,latino,209,0.0647
5,transgender,255,0.0624
6,atheist,123,0.0601
7,black,1496,0.0585
8,buddhist,49,0.0547
9,white,2551,0.0508


## 7. Counterfactual Gap

Replace each identity term with `"person"` (neutral substitution) and re-score the modified comment. The mean absolute score difference measures the *absolute* effect of mentioning an identity.

Pairwise swaps (e.g. `black` ↔ `white`) are added for comparison. Unlike neutral substitution, swap gaps cancel when both terms carry similar bias — neutral mode is the primary signal.

> **Note.** Only identity terms that appear as literal whole words in the text are matched (`black`, `white`, `muslim`, `christian`, `male`, `female`, etc.). Compound column names such as `homosexual_gay_or_lesbian` will not produce text matches and are silently skipped.

In [11]:
def predict_fn(texts: list[str]) -> np.ndarray:
    ids = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    model.eval()
    chunks: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(ids), BATCH_SIZE):
            chunks.append(model(ids[i : i + BATCH_SIZE].to(DEVICE)).cpu().numpy())
    return np.concatenate(chunks)

cf_results = evaluator.compute_counterfactual_gap(
    test_df,
    text_col  = "comment_text",
    score_col = "score",
    predict_fn = predict_fn,
    swap_pairs = [("black", "white"), ("christian", "muslim"), ("male", "female")],
)

cf_results.round(4)

,type,term_a,term_b,n_pairs,mean_gap,max_gap
0,neutral,black,person,1917,0.0821,0.8284
1,neutral,muslim,person,1046,0.0793,0.6821
2,neutral,transgender,person,160,0.0759,0.4505
3,neutral,white,person,3875,0.0616,0.7406
4,neutral,christian,person,991,0.0149,0.2435
5,neutral,jewish,person,339,0.0143,0.1890
6,neutral,hindu,person,33,0.0142,0.1510
7,neutral,buddhist,person,28,0.0106,0.0817
8,neutral,heterosexual,person,64,0.0100,0.1132
9,neutral,female,person,699,0.0081,0.2640


## Conclusion

**Metric averages across 18 subgroups.**

| Metric | Average |
|--------|---------|
| Subgroup AUC | 0.847 |
| BPSN AUC | 0.876 |
| BNSP AUC | 0.946 |
| Pinned AUC | 0.881 |
| FPR | 0.033 |
| FPR gap | 0.022 |
| Subgroup ECE | 0.052 |
| Neutral CF gap (14 terms) | 0.028 |

**Selected identity metrics.**

| Identity | BPSN AUC | FPR gap | ECE |
|----------|:--------:|:-------:|:---:|
| `black` | 0.806 | +0.059 | 0.059 |
| `homosexual_gay_or_lesbian` | 0.806 | +0.051 | 0.038 |
| `transgender` | 0.845 | +0.051 | 0.062 |
| `muslim` | 0.840 | +0.022 | 0.034 |
| `white` | 0.823 | +0.034 | 0.051 |
| `christian` | 0.932 | +0.003 | 0.015 |

Overall AUC 0.9540 · Predicted toxic rate 5.30% · Overall ECE 0.006

---

**Overall performance.** AUC = 0.9540, predicted toxic rate 5.30% vs. true 8.00%; recall 0.53, precision 0.80. The model under-flags in aggregate, but this masks severe over-flagging within specific identity subgroups.

**AUC.** Worst pinned AUC: `other_religion` (0.807, n=34) and `hindu` (0.814, n=55) — both noisy due to small size. Among reliable groups, `black` (0.844), `homosexual_gay_or_lesbian` (0.851), `white` (0.860), and `muslim` (0.865) are worst, all with low BPSN AUC (0.806–0.840): the model systematically over-predicts toxicity for non-toxic comments in these groups. No group shows notably low BNSP AUC; under-detection is not a primary failure mode.

**FPR.** `black` leads (7.03%, +5.88 pp above background), followed by `transgender` (+5.14 pp) and `homosexual_gay_or_lesbian` (+5.07 pp) — all three at roughly 5× the background rate. `white` (+3.43 pp) and `hindu` (+3.08 pp) are also well above background.

**ECE.** Overall ECE 0.006. Worst subgroups: `heterosexual` (0.115, ×19 overall) and `other_religion` (0.110, ×18) — notably distinct from the AUC/FPR worst groups. `transgender` (0.062), `black` (0.059), and `white` (0.051) are also poorly calibrated.

**Counterfactual gap.** `black → person` produces the largest mean score shift (0.082), with `muslim → person` (0.079) and `transgender → person` (0.076) close behind — roughly 10× the shift for lower-bias identities. The `muslim ↔ christian` swap is asymmetric (0.061 vs. 0.049), confirming a stronger token-level association between `muslim` and toxicity.

**Takeaway.** Five groups — `black`, `homosexual_gay_or_lesbian`, `transgender`, `white`, and `muslim` — concentrate most model bias across AUC and FPR. `transgender` was not highlighted in the EDA but emerges clearly in model predictions. Primary mitigations: label reweighting or upsampling for the five over-prediction groups; per-subgroup threshold calibration to close the FPR gap; separate calibration work for `heterosexual` and `other_religion` given their high ECE.